# 05 - Vegetation Response Analysis

Examines the relationship between hydroperiod persistence and vegetation
condition (NDVI) across the wetland using the full study-period dataset.

**Methods:**
- Full-period per-pixel hydroperiod persistence (fraction of all months inundated)
- Full-period per-pixel mean NDVI
- 2D log-density plot of NDVI vs persistence
- Binned regression (5% persistence bins) with 95% CI
- Linear vs piecewise linear model (AIC selection; piecewise preferred if ΔAIC > 10)
- Ecological risk zone map based on estimated threshold p*

**Inputs:** `inundation_cube.nc`, `ndvi_cube.nc`, `wetland_mask.tif`

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import rasterio
from scipy.stats import t as t_dist
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import BoundaryNorm, ListedColormap
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "figure.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

PROCESSED_DIR = Path("data/processed")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

ANALYSIS_START = "2019-01"
ANALYSIS_END   = "2024-12"

## Load Data and Compute Full-Period Variables

In [ ]:
# Load cubes
inundation_cube = xr.open_dataarray(PROCESSED_DIR / "inundation_cube.nc")
ndvi_cube       = xr.open_dataarray(PROCESSED_DIR / "ndvi_cube.nc")

inundation_cube = inundation_cube.sel(time=slice(ANALYSIS_START, ANALYSIS_END))
ndvi_cube       = ndvi_cube.sel(time=slice(ANALYSIS_START, ANALYSIS_END))

# Load wetland mask
with rasterio.open("data/raw/wetland_mask.tif") as src:
    mask_arr = src.read(1)
wetland_mask = xr.DataArray(
    mask_arr == 1, dims=["y", "x"],
    coords={"y": inundation_cube.y, "x": inundation_cube.x})

# Full-period hydroperiod persistence
valid = (inundation_cube >= 0) & wetland_mask
water = (inundation_cube == 1) & wetland_mask
persistence = xr.where(
    valid.sum("time") > 0,
    water.sum("time") / valid.sum("time").astype(float),
    np.nan).where(wetland_mask)

# Full-period mean NDVI within wetland mask
mean_ndvi = ndvi_cube.where(wetland_mask & (ndvi_cube > -1)).mean(dim="time")

print(f"Inundation cube: {inundation_cube.shape}")
print(f"NDVI cube:       {ndvi_cube.shape}")
print(f"Wetland pixels:  {int(wetland_mask.sum()):,}")
print(f"Mean persistence: {float(persistence.mean(skipna=True)):.3f}")
print(f"Mean NDVI:        {float(mean_ndvi.mean(skipna=True)):.3f}")

## Pixel-Level Extraction

In [ ]:
mask_flat    = wetland_mask.values.ravel()
persist_flat = persistence.values.ravel()
ndvi_flat    = mean_ndvi.values.ravel()

valid_px = mask_flat & ~np.isnan(persist_flat) & ~np.isnan(ndvi_flat)
p_vals   = persist_flat[valid_px] * 100   # convert to %
n_vals   = ndvi_flat[valid_px]

print(f"Valid pixels: {valid_px.sum():,}")
print(f"Persistence range: {p_vals.min():.1f}% – {p_vals.max():.1f}%")
print(f"NDVI range:        {n_vals.min():.3f} – {n_vals.max():.3f}")

## Figure 1 — NDVI vs Hydroperiod Persistence (Log Density)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

ndvi_min = np.floor(n_vals.min() * 10) / 10   # round down to nearest 0.1
ndvi_max = np.ceil(n_vals.max()  * 10) / 10

h, xedges, yedges = np.histogram2d(
    p_vals, n_vals, bins=[50, 50],
    range=[[0, 100], [ndvi_min, ndvi_max]])
h = np.ma.masked_where(h == 0, h)

im = ax.pcolormesh(xedges, yedges, np.log10(h.T + 1), cmap="YlOrRd", vmin=0)
cbar = plt.colorbar(im, ax=ax, label="log₁₀(pixel count + 1)")

ax.set_xlabel("Hydroperiod Persistence (%)")
ax.set_ylabel("Mean NDVI (full period)")
ax.set_xlim(0, 100)
ax.set_ylim(ndvi_min, ndvi_max)
ax.axhline(0, color="grey", linewidth=0.6, linestyle="--", alpha=0.5)
ax.set_title(
    "NDVI vs Hydroperiod Persistence — Wiang Nong Lom Wetland (2019–2024)",
    fontsize=11, fontweight="bold")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig_07_ndvi_density.pdf", bbox_inches="tight")
plt.savefig(OUTPUT_DIR / "fig_07_ndvi_density.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"NDVI range shown: {ndvi_min:.1f} to {ndvi_max:.1f}")
print("Saved: outputs/fig_07_ndvi_density.pdf/.png")

## Binned Regression: Linear vs Piecewise Linear

In [ ]:
from scipy import stats

def bin_ndvi_vs_persistence(p, n, bin_width=5):
    """Bin pixels for visualisation only — NOT used for model fitting."""
    bins = np.arange(0, 101, bin_width)
    ctrs, means, ci_lo, ci_hi, counts = [], [], [], [], []
    for lo, hi in zip(bins[:-1], bins[1:]):
        idx = (p >= lo) & (p < hi)
        sub = n[idx]
        if len(sub) < 5:
            continue
        m  = np.nanmean(sub)
        se = np.nanstd(sub, ddof=1) / np.sqrt(len(sub))
        tc = t_dist.ppf(0.975, df=len(sub) - 1)
        ctrs.append((lo + hi) / 2)
        means.append(m)
        ci_lo.append(m - tc * se)
        ci_hi.append(m + tc * se)
        counts.append(len(sub))
    return (np.array(ctrs), np.array(means),
            np.array(ci_lo), np.array(ci_hi), np.array(counts))


def fit_linear_raw(x, y):
    """OLS linear fit on raw pixel data. AIC uses n = len(x)."""
    slope, intercept, r, p, _ = stats.linregress(x, y)
    y_pred = intercept + slope * x
    rss = np.sum((y - y_pred) ** 2)
    n, k = len(x), 2
    return dict(intercept=intercept, slope=slope, r_squared=r**2,
                rss=rss, aic=n * np.log(rss / n) + 2 * k, y_pred=y_pred)


def fit_piecewise_raw(x, y, bp_range=(10, 90), bp_step=1):
    """
    Fit continuous piecewise linear on raw pixel data.
    Grid search at bp_step resolution. AIC uses n = len(x).
    """
    best_aic, best = np.inf, None
    n, k = len(x), 4   # a1, b1, b2, bp
    for bp in np.arange(bp_range[0], bp_range[1] + bp_step, bp_step):
        left  = x <  bp
        right = x >= bp
        if left.sum() < 5 or right.sum() < 5:
            continue
        sl = stats.linregress(x[left], y[left])
        a1, b1 = sl.intercept, sl.slope
        y_bp = a1 + b1 * bp
        # OLS through origin for right segment (continuity enforced)
        dx = x[right] - bp
        b2 = np.sum(dx * (y[right] - y_bp)) / np.sum(dx ** 2)
        y_pred = np.where(left, a1 + b1 * x, y_bp + b2 * (x - bp))
        rss = np.sum((y - y_pred) ** 2)
        aic = n * np.log(rss / n) + 2 * k
        if aic < best_aic:
            best_aic = aic
            best = dict(breakpoint=bp, a1=a1, b1=b1, b2=b2,
                        y_bp=y_bp, rss=rss, aic=aic)
    return best


# Bin for visualisation
ctrs, means, ci_lo, ci_hi, counts = bin_ndvi_vs_persistence(p_vals, n_vals)

# Fit models to raw pixels
print("Fitting linear model on raw pixel data...")
lin = fit_linear_raw(p_vals, n_vals)

print("Fitting piecewise model (1% grid search, 10–90%)...")
pw  = fit_piecewise_raw(p_vals, n_vals, bp_range=(10, 90), bp_step=1)

delta_aic   = lin["aic"] - pw["aic"]
use_piecewise = delta_aic > 10
p_star      = pw["breakpoint"]

print(f"\nModel comparison (n = {len(p_vals):,} wetland pixels)")
print(f"  Linear    AIC = {lin['aic']:.1f},  R² = {lin['r_squared']:.4f}")
print(f"  Piecewise AIC = {pw['aic']:.1f},  p* = {p_star:.0f}%")
print(f"  ΔAIC = {delta_aic:.1f} → {'piecewise' if use_piecewise else 'linear'} preferred")

## Figure 2 — Binned NDVI–Persistence with Model Fit

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

# CI ribbon and binned means
ax.fill_between(ctrs, ci_lo, ci_hi, alpha=0.18, color="#2c7bb6", label="95% CI")
ax.plot(ctrs, means, "o", color="#2c7bb6", markersize=5,
        label="Binned mean NDVI (5% bins, visualisation only)")

# Pixel-count bars on secondary axis
ax2 = ax.twinx()
ax2.bar(ctrs, counts / 1000, width=4, alpha=0.12, color="grey")
ax2.set_ylabel("Pixel count (×10³)", color="grey", fontsize=9)
ax2.tick_params(axis="y", labelcolor="grey", labelsize=8)
ax2.set_ylim(0, counts.max() / 1000 * 6)
ax2.spines["top"].set_visible(False)

# Model fits on smooth grid
x_s = np.linspace(0, 100, 500)
y_lin = lin["intercept"] + lin["slope"] * x_s
ax.plot(x_s, y_lin, "--", color="grey", linewidth=1.5,
        label=f'Linear (AIC={lin["aic"]:.0f}, R²={lin["r_squared"]:.3f})')

y_pw = np.where(
    x_s < p_star,
    pw["a1"] + pw["b1"] * x_s,
    pw["y_bp"] + pw["b2"] * (x_s - p_star))
ax.plot(x_s, y_pw, "-", color="#d7191c", linewidth=2.2,
        label=f'Piecewise (AIC={pw["aic"]:.0f}, p*={p_star:.0f}%, ΔAIC={delta_aic:.0f})')
ax.axvline(p_star, color="#d7191c", linestyle="--", linewidth=1.4, alpha=0.7)

# Y-axis spans full data range
y_margin = (n_vals.max() - n_vals.min()) * 0.05
ax.set_ylim(n_vals.min() - y_margin, n_vals.max() + y_margin)
ax.text(p_star + 1, n_vals.min(),
        f"p* = {p_star:.0f}%", color="#d7191c", fontsize=9, va="bottom")

ax.set_xlabel("Hydroperiod Persistence (%)")
ax.set_ylabel("Mean NDVI")
ax.set_xlim(0, 100)
ax.legend(fontsize=8.5, loc="upper right")
ax.grid(True, alpha=0.25)
ax.set_title(
    "NDVI–Hydroperiod Persistence — Wiang Nong Lom Wetland (2019–2024)\n"
    f"Models fitted to raw pixels (n = {len(p_vals):,}); bins shown for visualisation",
    fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig_08_ndvi_persistence_model.pdf", bbox_inches="tight")
plt.savefig(OUTPUT_DIR / "fig_08_ndvi_persistence_model.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: outputs/fig_08_ndvi_persistence_model.pdf/.png")

## Figure 3 — Ecological Risk Zone Map

In [ ]:
# Risk classification on full-period persistence map
# Two zones only — threshold is the model-derived p*
# Safe:     persistence <  p*  (below NDVI decline threshold)
# Critical: persistence >= p*  (exceeds threshold associated with vegetation decline)

p_map   = persistence.values * 100
in_mask = wetland_mask.values & ~np.isnan(p_map)

risk_map = np.full(p_map.shape, np.nan)
risk_map[in_mask & (p_map <  p_star)] = 0   # Safe
risk_map[in_mask & (p_map >= p_star)] = 1   # Critical

n_safe  = (risk_map == 0).sum()
n_crit  = (risk_map == 1).sum()
n_total = in_mask.sum()

print(f"Risk zones (full period, p* = {p_star:.1f}%):")
print(f"  Safe     (<  {p_star:.1f}%): {n_safe:,} px ({n_safe/n_total*100:.1f}%)")
print(f"  Critical (>= {p_star:.1f}%): {n_crit:,} px ({n_crit/n_total*100:.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

cmap_risk = ListedColormap(["#2ca25f", "#d73027"])
norm_risk = BoundaryNorm([-0.5, 0.5, 1.5], cmap_risk.N)

x = persistence.x.values; y = persistence.y.values
dx = abs(x[1]-x[0])/2;   dy = abs(y[1]-y[0])/2
ext = [x[0]-dx, x[-1]+dx, y[-1]-dy, y[0]+dy]

im0 = axes[0].imshow(risk_map, cmap=cmap_risk, norm=norm_risk,
                     extent=ext, origin="upper", aspect="equal")
cbar = plt.colorbar(im0, ax=axes[0], ticks=[0, 1], shrink=0.8)
cbar.set_ticklabels([
    f"Safe (< {p_star:.0f}%)",
    f"Critical (≥ {p_star:.0f}%)"])
axes[0].set_title(f"Vegetation stress zone\n(threshold = model-derived p* = {p_star:.0f}%)")
axes[0].set_xlabel("Easting (m)")
axes[0].set_ylabel("Northing (m)")
axes[0].ticklabel_format(style="sci", axis="both", scilimits=(0,0))

labels = [
    f"Safe\n(< {p_star:.0f}%)\n{n_safe/n_total*100:.1f}%",
    f"Critical\n(≥ {p_star:.0f}%)\n{n_crit/n_total*100:.1f}%"]
wedges, texts = axes[1].pie(
    [n_safe, n_crit],
    labels=labels, colors=["#2ca25f", "#d73027"],
    startangle=90, wedgeprops={"edgecolor": "white", "linewidth": 1.5})
for t in texts: t.set_fontsize(10)
axes[1].set_title("Area by Vegetation stress zone")

plt.suptitle(
    f"Ecological Risk Assessment — Wiang Nong Lom Wetland (2019–2024)\n"
    f"Threshold p* = {p_star:.0f}% derived from piecewise AIC model selection",
    fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig_09_risk_zones.pdf", bbox_inches="tight")
plt.savefig(OUTPUT_DIR / "fig_09_risk_zones.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: outputs/fig_09_risk_zones.pdf/.png")

## Summary

| Parameter | Value |
|-----------|-------|
| Persistence threshold p* | model-derived (piecewise AIC) |
| Safe zone | persistence < p* |
| Critical zone | persistence ≥ p* |

p* is the only threshold used, directly estimated from the data via AIC model selection.

**Next:** `06_landscape_metrics.ipynb`